# Rough American Hybrid Pricer 项目答辩讲解

本 notebook 用来从答辩角度讲清楚整个项目：项目目标、模块设计、数学逻辑、实验产出、结果解释，以及真实市场数据应用验证。

项目主线不是“做一个交易机器人”，而是：

> 构建一个面向美式期权的混合定价器，融合经典模型、LSMC、rough volatility、元模型和不确定性校准，并用真实期权链做错价检测应用验证。

## 1. 项目一句话介绍

本项目实现了一个 **Hybrid American Option Pricing Framework**。

它做的事情是：

1. 用 Black-Scholes、Binomial Tree、Monte Carlo 建立基础定价器。
2. 用 Longstaff-Schwartz Monte Carlo 解决美式期权提前行权问题。
3. 用 rough Bergomi-style stochastic volatility 模拟更真实的粗糙波动率路径。
4. 生成定价数据集，用高精度 rough Bergomi LSMC 作为 target。
5. 训练混合元模型，融合多个基础定价器输出。
6. 用 MC Dropout 和 conformal calibration 输出不确定性区间。
7. 接入真实历史期权链，做 market mispricing detection 应用验证。

答辩时可以这样说：

> 我的核心目标不是直接构建实盘交易系统，而是构建一个可解释、可比较、带不确定性估计的美式期权混合定价器。真实市场数据和回测模块只是为了验证定价器输出能否产生合理的市场错价信号。

## 2. 项目整体架构

项目可以分成五层：

```text
options/        期权合约、市场状态、payoff
models/         各类定价器：BS、Tree、MC、LSMC、rough Bergomi
meta/           特征工程、MC Dropout、校准方法
data/           真实/样本期权链接入和统一 schema
strategy/       fair value 和 mispricing signal
backtest/       应用验证：简单历史信号检验
experiments/    每个阶段的可运行实验脚本
reports/        输出的表格和图片
tests/          单元测试和 smoke tests
```

架构思想是：

```text
期权定义 -> 路径模拟/定价器 -> 元模型融合 -> 不确定性校准 -> 市场数据应用验证
```

这样做的好处是，每个模块职责清楚：定价器不关心数据源，数据源不关心策略，策略只消费统一 schema 和 fair value。

## 3. 阶段 1：经典基准定价器

阶段 1 的目标是建立 baseline。

实现了：

- Black-Scholes 欧式期权闭式解
- Cox-Ross-Rubinstein Binomial Tree
- 普通 Monte Carlo 欧式期权定价

为什么要做这些基准？

因为后面 LSMC、rough Bergomi、元模型都更复杂。没有基准，就不知道复杂模型是否合理。

重点比较：

```text
Black-Scholes vs Binomial Tree
Monte Carlo vs Black-Scholes
American Put vs European Put
```

这一步能验证：

- Tree 对欧式期权应接近 Black-Scholes。
- Monte Carlo 会接近 Black-Scholes，但有随机误差。
- American Put 通常不低于 European Put，因为美式期权多了提前行权权利。

产出：

```text
reports/tables/benchmark_results.csv
reports/figures/benchmark_runtime.png
```

## 4. 阶段 2：LSMC 美式期权定价

美式期权难点是可以提前行权，所以不能只看最终到期 payoff。

我们把问题建模为 **optimal stopping problem**：

```text
每个时间点都要决定：现在行权，还是继续持有？
```

Longstaff-Schwartz Monte Carlo 的逻辑：

1. 正向模拟很多条资产价格路径。
2. 从到期日往前倒推。
3. 每个时间点只看价内路径。
4. 用回归估计 continuation value，也就是继续持有的价值。
5. 比较 immediate exercise payoff 和 continuation value。
6. 如果现在行权收益更高，就在这里行权。
7. 最后把每条路径的现金流折现回今天求平均。

回归基函数支持：

```text
polynomial
Laguerre
Hermite
```

LSMC 的价值在于：它能处理路径依赖和提前行权，是美式期权 Monte Carlo 定价的核心方法。

产出：

```text
reports/tables/lsmc_convergence.csv
reports/tables/lsmc_basis_comparison.csv
reports/tables/lsmc_exercise_boundary.csv
reports/figures/lsmc_convergence.png
reports/figures/lsmc_exercise_boundary.png
```

## 5. LSMC 回归的大白话解释

回归的目标不是预测股票价格，而是预测：

```text
如果我现在不行权，未来这条路径大概还能值多少钱？
```

例如某个时间点有三条价内路径：

```text
当前股价 S:       80, 90, 95
未来折现现金流 y: 20, 12,  8
```

我们用这些点拟合一个函数：

```text
continuation_value = f(S)
```

然后对每条路径比较：

```text
现在行权收益 = K - S
继续持有价值 = f(S)
```

如果 `K - S > f(S)`，就现在行权。

这就是 LSMC 的核心：用路径样本和回归近似一个本来很难直接算的 continuation value。

## 6. 阶段 3：rough Bergomi-style 随机波动率

真实市场的波动率不是常数，也不是非常平滑的曲线。实证研究发现波动率有“粗糙性”：短时间尺度下变化很剧烈。

因此我们加入 rough Bergomi-style path simulation。

核心参数：

```text
H     Hurst 指数，越小波动率越粗糙
eta   vol-of-vol，控制波动率本身波动多大
rho   股票收益和波动率冲击的相关性
xi0   初始方差水平
```

这一步做了：

1. 生成 fractional Gaussian noise。
2. 模拟 rough variance path。
3. 用随机方差驱动资产价格路径。
4. 把 rough paths 接入 MC 和 LSMC。
5. 比较常数波动率和 rough volatility 下的价格差异。

项目亮点：rough Bergomi 让项目不只是普通美式期权定价，而是加入了更现代的随机波动率研究特色。

产出：

```text
reports/figures/rough_bergomi_volatility_paths.png
reports/figures/rough_bergomi_asset_paths.png
reports/figures/hurst_sensitivity.png
reports/tables/rough_bergomi_price_table.csv
reports/tables/hurst_sensitivity.csv
```

## 7. 阶段 4：数据集生成与 target 构造

元模型训练需要数据集。我们随机采样期权参数空间：

```text
S0, K, T, r, sigma, dividend
H, eta, rho
```

每一行样本包含：

```text
market parameters
rough volatility parameters
base pricer outputs
target price
market realism fields
```

重要设计：target 不再是普通二叉树，而是：

```text
high-fidelity rough Bergomi LSMC
```

为什么？

因为项目核心是 rough volatility + American option，所以 target 应尽量贴近项目主线，而不是只用经典 Tree。

新增市场现实字段：

```text
bid, ask, mid_price
spread, relative_spread
market_iv, model_iv, iv_error
moneyness, log_moneyness
maturity bucket
dividend yield
```

产出：

```text
data/processed/pricing_dataset.csv
data/processed/train.csv
data/processed/valid.csv
data/processed/test.csv
reports/figures/pricing_dataset_distributions.png
```

## 8. 阶段 5：混合元模型

混合元模型的想法是：

```text
不要只相信一个定价器，而是让模型学习如何融合多个定价器。
```

输入特征包括：

```text
market parameters
rough volatility parameters
Black-Scholes price
Binomial Tree price
Monte Carlo price
LSMC price
rough Bergomi price
moneyness / maturity / dividend
```

训练模型：

```text
Linear Regression
Random Forest
Gradient Boosting
MC Dropout Neural Network
```

评价指标：

```text
MAE
RMSE
MAPE
```

这里的核心不是证明某个模型永远最好，而是比较：

```text
混合模型是否能接近或超过单一定价器？
哪些特征最重要？
模型在什么区域误差更大？
```

产出：

```text
reports/tables/meta_model_results.csv
reports/tables/meta_model_predictions.csv
reports/tables/meta_model_feature_importance.csv
reports/figures/meta_model_prediction_vs_true.png
reports/figures/meta_model_residuals.png
```

## 9. 阶段 6：不确定性区间与校准

传统定价器通常只给一个价格：

```text
price = 7.21
```

但真实建模中，我们还想知道模型有多确定：

```text
price = 7.21
95% interval = [6.65, 7.94]
```

本项目用 MC Dropout 生成预测分布。

大白话：

```text
预测时也随机关闭一部分神经元，重复预测很多次。
这些预测值形成一个分布。
均值作为点预测，分位数作为区间。
```

然后用 conformal calibration 修正区间。

为什么需要校准？

因为原始模型区间可能太窄，覆盖率不足。conformal 方法用 validation set 看模型通常错多少，再把区间适当放宽。

产出：

```text
reports/tables/uncertainty_results.csv
reports/tables/uncertainty_predictions.csv
reports/tables/uncertainty_calibration_curve.csv
reports/figures/uncertainty_calibration_curve.png
reports/figures/uncertainty_interval_widths.png
reports/figures/uncertainty_calibrated_intervals.png
```

## 10. 阶段 7：真实期权链数据接入

为了让项目不只停留在模拟数据，我们加入真实 option chain data interface。

统一 schema：

```text
ticker
quote_date
expiration
strike
option_kind
bid
ask
mid_price
market_iv
volume
open_interest
underlying_price
rate
dividend
```

支持的数据源：

```text
sample option chain
local CSV
yfinance current option chain
ORATS historical EOD option chain
OnclickMedia free historical option chain
```

为什么要统一 schema？

因为后面的定价器、错价信号、应用验证都不应该关心数据来自哪里。只要数据被标准化成同一张表，后续模块就能复用。

## 11. 阶段 8：Market Mispricing Application

这一部分是应用验证，不是项目主线。

它要回答的是：

```text
我们的定价器能否连接真实市场 option chain，生成可解释的 mispricing signal？
```

信号构造：

```text
buy_edge  = fair_value - ask
sell_edge = bid - fair_value
```

如果：

```text
fair_value > ask
```

说明模型认为市场 ask 偏低，可能 underpriced。

如果：

```text
fair_value < bid
```

说明模型认为市场 bid 偏高，可能 overpriced。

回测模块只是用来检查这些信号在历史上是否有经济意义。它不是生产交易系统，也不是项目主要贡献。

## 12. 为什么交易模块不能喧宾夺主

本项目主题是：

```text
American option hybrid pricing
```

不是：

```text
high-frequency trading strategy
production market-making system
guaranteed profitable strategy
```

所以交易相关模块应定位为：

```text
真实市场应用验证
```

答辩时可以这样讲：

> 我没有把回测收益作为项目唯一目标，因为定价误差不等于可交易 alpha。交易结果还受到 bid-ask spread、流动性、方向风险、IV regime、theta decay 和数据质量影响。因此我把交易模块作为应用验证，用来证明定价器可以接入真实市场数据并生成可解释的错价信号。

## 13. 当前项目主要结论

可以总结为以下几点：

1. 经典基准模型验证了代码基础：Tree 接近 Black-Scholes，MC 有随机误差，American Put 存在提前行权价值。
2. LSMC 成功把美式期权定价建模为最优停止问题，并能提取提前行权边界。
3. rough Bergomi-style 模块展示了 Hurst 指数越小，波动率路径越粗糙，并且 rough volatility 会影响期权价格。
4. 数据集构造从普通 tree target 升级成 high-fidelity rough Bergomi LSMC target，更贴合项目主题。
5. 元模型能融合多个基础定价器，并输出点预测误差指标和特征重要性。
6. MC Dropout + conformal calibration 让模型不只给价格，还给不确定性区间。
7. 市场数据层可以接入真实 option chain，并把不同来源统一成同一 schema。
8. Market mispricing application 证明了定价器输出可以和真实 bid/ask 结合，形成可解释的错价检测流程。

## 14. 项目局限性

答辩时主动讲局限性会显得更专业。

当前局限包括：

1. 元模型主要训练在 synthetic rough-volatility dataset 上，还不是用真实市场 mid price 训练。
2. rough Bergomi 实现是 rough Bergomi-style，用于研究展示，不是工业级校准系统。
3. MC Dropout 是近似贝叶斯方法，不是完整 Bayesian neural network posterior inference。
4. 免费历史期权数据需要进一步检查覆盖率、缺失、异常报价和 IV 质量。
5. Mispricing backtest 是应用验证，不代表可直接实盘。
6. 真实市场还需要考虑手续费、滑点、保证金、提前行权风险、财报、分红和 regime shift。

这些局限不是项目失败，而是说明你知道模型和真实市场之间的边界。

## 15. 如果继续扩展

后续可以继续做：

```text
1. 用真实 option chain mid price 训练 market-calibrated meta-model
2. 加入 Greeks 和 IV surface 特征
3. 做数据质量报告 data_quality_report.csv
4. 对 rough Bergomi 参数做市场校准
5. 加入更严格的 out-of-sample validation
6. 做 Streamlit dashboard 展示 fair value、bid/ask、uncertainty interval 和 mispricing signal
```

但这些是扩展方向，当前项目已经形成完整闭环：

```text
定价理论 -> 模型实现 -> 数据集 -> 元模型 -> 不确定性 -> 真实市场应用验证
```

## 16. 答辩结尾总结

可以用这段话收尾：

> 本项目围绕美式期权定价中的提前行权和随机波动率问题，构建了从经典基准、LSMC、rough Bergomi-style stochastic volatility 到混合元模型和不确定性校准的完整定价框架。与单一定价器相比，混合框架能够同时利用多个模型的定价信息，并通过 MC Dropout 与 conformal calibration 给出预测区间。最后，我将该定价器接入真实历史 option chain，用 market fair value 和 bid/ask 的偏差构造 mispricing signal，作为真实市场应用验证。项目重点不在于宣称实盘盈利，而在于展示一个可解释、可扩展、带不确定性估计的美式期权混合定价系统。